# NLLB-200: Yoruba → English Idiom Translation
Machine Translation of Yoruba Idiomatic Expressions  
**Model:** NLLB-200 Distilled 600M

**Setting:** Direct Translation (Yoruba → English)  
**Metrics:** BLEU · chrF+ · BERTScore F1


## 1. Install Dependencies and Load Model

In [1]:
!pip install transformers torch sacrebleu bert-score -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"Model loaded on: {device}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded on: cuda


## 2. Mount Google Drive and Load Dataset

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/Yoruba Idioms.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATASET_PATH)
df.columns = ['idiom', 'literal_meaning', 'idiomatic_meaning']
df = df.dropna(subset=['idiom', 'idiomatic_meaning']).reset_index(drop=True)

print(f"Dataset loaded: {len(df)} samples")
print(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded: 104 samples
                   idiom                                 literal_meaning  \
0      Ta téru nípàá                          Kick the calico fabric   
1           Júbà ehoro                          Pay homage to the hare   
2         Fi àáké kọ́rí                         Hang an axe on the head   
3  Tutọ́ sókè fojú gbà á  Spit upward and let it land back on your face,   
4            Ru igi Oyin        Carry a tree with honey bees on the head   

             idiomatic_meaning  
0                       To die  
1                  To run away  
2            Insist completely  
3        Be excessively angry.  
4  Throw oneself into trouble.  


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Translation Function

In [5]:
def translate_yo_to_en(text: str) -> str:
    """
    Translates a Yorùbá idiom to English using NLLB-200.

    NLLB language codes:
        Yorùbá  → yor_Latn
        English → eng_Latn

    Args:
        text: Yorùbá idiomatic expression

    Returns:
        English translation string
    """
    # Set source language to Yorùbá
    tokenizer.src_lang = "yor_Latn"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    #English as target language
    target_lang_id = tokenizer.convert_tokens_to_ids("eng_Latn")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_new_tokens=100,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Translation function defined")


Translation function defined


## 4. Run NLLB-200

In [6]:
from tqdm import tqdm

test_df      = df.copy()
predictions  = []

print(f"Running NLLB-200 on {len(test_df)} samples ...\n")

for idiom in tqdm(test_df['idiom'], desc="YO → EN"):
    try:
        pred = translate_yo_to_en(idiom)
    except Exception as e:
        print(f"[Error] {idiom[:30]}... → {e}")
        pred = ""
    predictions.append(pred)

# Attach predictions to the dataframe
test_df['nllb_prediction'] = predictions

print(f"\ncomplete")
print(f"   Samples translated: {len(predictions)}")


Running NLLB-200 on 104 samples ...



YO → EN: 100%|██████████| 104/104 [00:20<00:00,  4.98it/s]


complete
   Samples translated: 104


## 5. Automatic Evaluation

In [7]:
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score_fn

def evaluate(predictions: list, references: list, label: str) -> dict:
    """
    Evaluate NLLB-200 predictions against idiomatic reference meanings.

    Metrics:
        BLEU      — n-gram precision (Papineni et al., 2002)
        chrF+     — character n-gram F-score with word unigrams (Popović, 2017)
        BERTScore — semantic similarity via contextual embeddings (Zhang et al., 2020)

    Args:
        predictions : list of NLLB-200 translations
        references  : list of reference idiomatic meanings
        label       : evaluation label string

    Returns:
        dict of metric scores
    """
    bleu      = BLEU()
    chrf_plus = CHRF(word_order=1)

    bleu_score      = bleu.corpus_score(predictions, [references]).score
    chrf_plus_score = chrf_plus.corpus_score(predictions, [references]).score
    _, _, F1        = bert_score_fn(predictions, references, lang="en", verbose=False)

    return {
        "Setting"      : label,
        "BLEU"         : round(bleu_score, 4),
        "chrF+"        : round(chrf_plus_score, 4),
        "BERTScore_F1" : round(F1.mean().item(), 4)
    }

# Evaluate against idiomatic meaning (figurative reference)
references  = test_df['idiomatic_meaning'].tolist()
results     = evaluate(test_df['nllb_prediction'].tolist(), references, "NLLB-200 (Direct)")

results_df  = pd.DataFrame([results])

print("\n" + "=" * 60)
print("        NLLB-200 EVALUATION RESULTS — YO → EN")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



        NLLB-200 EVALUATION RESULTS — YO → EN
          Setting   BLEU   chrF+  BERTScore_F1
NLLB-200 (Direct) 0.5579 10.0003        0.8653


## 6. Save and Download Results

In [8]:
from google.colab import files
predictions_path = os.path.join(OUTPUT_DIR, "nllb_predictions.csv")
test_df.to_csv(predictions_path, index=False)
print(f"Predictions saved → {predictions_path}")
print(f"   Columns : {list(test_df.columns)}")

# Save Metrics CSV
metrics_path = os.path.join(OUTPUT_DIR, "nllb_metrics.csv")
results_df.to_csv(metrics_path, index=False)
print(f"Metrics saved    → {metrics_path}")

# Download both files
print("\nDownloading files ...")
files.download(predictions_path)
files.download(metrics_path)

print("\nAll files downloaded")


Predictions saved → /content/drive/MyDrive/results/nllb_predictions.csv
   Columns : ['idiom', 'literal_meaning', 'idiomatic_meaning', 'nllb_prediction']
Metrics saved    → /content/drive/MyDrive/results/nllb_metrics.csv



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All files downloaded
